# Section 7 — Foundry Evaluations 

## Storyline

Athora Netherlands' compliance team won't sign off PolicyPal on the strength of *"it looks good in DevUI"*. They want a number: **on a representative set of rep questions, what % does PolicyPal answer correctly, helpfully, and without hallucination or PII leakage?** And they want that number re-runnable every time you change the prompt or model.

That's evaluation.

## What you'll do

| Step | Concept | Why it matters for Athora Netherlands |
|------|---------|---------------------------|
| 1 | A small Athora Netherlands dataset (`eval_dataset`) | Without ground truth, no evaluation. |
| 2 | Local evaluators: `keyword_check`, `tool_calls_present`, custom `@evaluator` | Cheap, fast, no extra service. Run on every PR. |
| 3 | Expected-output comparison | Catch regressions when the prompt changes. |
| 4 | Foundry-grade evaluators (relevance, groundedness, coherence, safety) | The numbers compliance actually wants. (Conceptual walk-through; we'll show one if time permits.) |
| 5 | Wire eval into a CI gate | `raise_for_status()` fails the build when scores drop. |

## PolicyPal under test

We need a deterministic-ish PolicyPal for evaluation. Real Athora Netherlands teams would use a frozen prompt + low temperature for evaluation runs.

In [ ]:
import os
from typing import Annotated
from agent_framework import Agent, tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from pydantic import Field

load_dotenv()

POLICIES = {
    "NL-2031-887": {
        "holder": "Jan de Vries",
        "product": "Pension - defined contribution",
        "balance_eur": 78_420.55,
        "monthly_contribution_eur": 350.00,
        "inception_date": "2014-06-01",
    },
    "NL-4408-552": {
        "holder": "Sanne Bakker",
        "product": "Life insurance - term",
        "balance_eur": 0.0,
        "monthly_contribution_eur": 42.50,
        "inception_date": "2019-11-15",
    },
}

@tool(approval_mode="never_require")
def get_policy(policy_number: Annotated[str, Field(description="Policy number")]) -> str:
    """Look up an Athora Netherlands policy by its number."""
    p = POLICIES.get(policy_number.upper())
    return f"Policy {policy_number}: {p}" if p else f"No policy found for {policy_number}."

client = FoundryChatClient(
    project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    model=os.environ["FOUNDRY_MODEL"],
    credential=AzureCliCredential(),
)

policypal = Agent(
    client=client,
    name="policypal-eval",
    id="policypal-eval",
    instructions=(
        "You are PolicyPal, an internal assistant for Athora Netherlands reps. Always use get_policy for any policy fact. "
        "If the policy is not found, say so plainly. Never invent figures."
    ),
    tools=[get_policy],
)
print("PolicyPal under test ready.")

### The dataset

Five questions a real Athora Netherlands rep might ask, with expected key facts. In production, this would be a versioned `.jsonl` file owned by QA and compliance. Reference: [`microsoft/agent-framework — python/samples/evaluation`](https://github.com/microsoft/agent-framework/tree/main/python/samples/evaluation).

In [ ]:
eval_dataset = [
    {"q": "What product is on policy NL-2031-887?", "expected": "defined contribution pension Jan de Vries"},
    {"q": "What is the balance on NL-2031-887?", "expected": "78420.55"},
    {"q": "Who is the holder of NL-4408-552?", "expected": "Sanne Bakker"},
    {"q": "What is the monthly premium on NL-4408-552?", "expected": "42.50 term life"},
    {"q": "What's on policy NL-9999-000?", "expected": "no policy found"},
]
queries = [d["q"] for d in eval_dataset]
expected = [d["expected"] for d in eval_dataset]
print(f"{len(queries)} eval cases ready.")

## Local evaluators (no extra service)

`LocalEvaluator` runs deterministic checks — keyword presence, custom Python functions, tool-call presence. No model-as-judge cost; runs in seconds; cheap to put in CI.

In [ ]:
from agent_framework import (
    LocalEvaluator,
    evaluate_agent,
    evaluator,
    tool_calls_present,
)

@evaluator
def is_concise(response: str) -> bool:
    """Reject answers longer than ~80 words — reps want short."""
    return len(response.split()) <= 80

@evaluator
def no_pii_leaked(response: str) -> bool:
    """Fail if the answer exposes unnecessary personal contact or national-ID style data."""
    forbidden_markers = ["BSN", "passport", "email:", "phone:", "address:"]
    return not any(marker.lower() in response.lower() for marker in forbidden_markers)

local = LocalEvaluator(
    is_concise,
    no_pii_leaked,
    tool_calls_present,
)

results = await evaluate_agent(
    agent=policypal,
    queries=queries,
    expected_output=expected,
    evaluators=local,
)

for r in results:
    print(f"\n=== {r.provider}: {r.passed}/{r.total} passed ===")
    for item in r.items:
        print(f"  Q: {item.input_text}")
        print(f"  A: {item.output_text[:120]}")
        for s in item.scores:
            print(f"    {'PASS' if s.passed else 'FAIL'} {s.name}")

### What just happened

- `evaluate_agent` ran PolicyPal once per query and ran every evaluator on every response.
- Every evaluator returned a `bool` (or a 0..1 float) — that became a per-item score.
- `tool_calls_present` failed for any query where PolicyPal answered without calling `get_policy` — useful signal.

## Foundry-grade evaluators (Task Adherence, Coherence, Safety)

These use a **model-as-judge** via the Foundry evaluation API. Unlike local checks, they assess *intent*, *quality*, and *safety* — which is what compliance actually wants.

Notice the contrast: with local evaluators you wrote Python functions for every check. With Foundry evaluators you just **pick names from a list** — the model-as-judge handles the rest.

Reference: [Evaluate your AI agents](https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/evaluate-agent)


In [ ]:
from agent_framework.foundry import FoundryEvals, FoundryChatClient

# Same client setup as before
judge_client = FoundryChatClient(
    project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    model=os.environ["FOUNDRY_MODEL"],
    credential=AzureCliCredential(),
)

# This is it — just pick evaluator names. No custom Python logic needed.
foundry_evals = FoundryEvals(
    client=judge_client,
    evaluators=[
        FoundryEvals.TASK_ADHERENCE,
        FoundryEvals.INTENT_RESOLUTION,
        FoundryEvals.TOOL_SELECTION,
        FoundryEvals.TOOL_INPUT_ACCURACY,
        FoundryEvals.RELEVANCE,
        FoundryEvals.COHERENCE,
        FoundryEvals.FLUENCY,
        FoundryEvals.VIOLENCE,
    ],
)

# Same evaluate_agent() call — just swap the evaluator
foundry_results = await evaluate_agent(
    agent=policypal,
    queries=queries,
    evaluators=foundry_evals,
)

r = foundry_results[0]
print(f"=== {r.provider}: {r.passed}/{r.total} passed ===")
print(f"  Counts: {r.result_counts}")
print(f"  Report: {r.report_url}\n")

for i, item in enumerate(r.items):
    q = item.input_text or queries[i]
    scores = "  ".join(
        f"{'PASS' if s.passed else 'FAIL'} {s.name}={s.score}" for s in item.scores
    )
    print(f"  Q: {q}")
    print(f"  A: {item.output_text[:120]}")
    print(f"    {scores}\n")

## CI gate with raise_for_status

`raise_for_status()` works on **both** local and Foundry eval results — call it on any result set to turn eval into a quality gate. If any item fails, the script exits non-zero and the CI build is red.

In [ ]:
# Toggle to True to demonstrate the CI gate behaviour
ENFORCE = False
if ENFORCE:
    for r in results + foundry_results:
        r.raise_for_status()  # raises EvalNotPassedError on first failure
else:
    print("CI gate disabled for the workshop — flip ENFORCE to True to test it.")

## Recap and what's next

- `evaluate_agent(agent, queries, evaluators=...)` is the one entry point — local + Foundry evaluators plug into it.
- Local evaluators = fast but not out of the box functionality.
- Foundry evaluators = model-graded quality (relevance, groundedness, safety) for release

